# Radar Basics: step-by-step coding notebook

这个 notebook 会按小步推进的方式学习 `radar_basics` codebase。每一步我们只写少量代码，先运行、观察，再解释它在 radar simulation mental model 里代表什么。

## Step 1: YAML 是实验说明书

先不要急着进入 `RadarSystem` 或信号处理。这个项目的入口是一份 YAML config：它描述一次雷达仿真实验需要哪些东西。

这一步我们只做三件事：

1. 找到 `example_configs/architecture.yaml`。
2. 把 YAML 读成 Python dict。
3. 看清楚它有哪些 top-level sections。

Mental model: YAML 不是仿真结果，也不是雷达对象本身；它是一次 experiment 的 declarative specification。

In [1]:
from pathlib import Path
from pprint import pprint

import yaml


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root from either the repo folder or the learning folder."""
    start = start or Path.cwd()
    for candidate in (start, *start.parents):
        config_path = candidate / "example_configs" / "architecture.yaml"
        if (candidate / "pyproject.toml").exists() and config_path.exists():
            return candidate
    raise FileNotFoundError("Could not find the radar_basics project root")


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "example_configs" / "architecture.yaml"

with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    raw_config = yaml.safe_load(handle)

print(f"Project root: {PROJECT_ROOT}")
print(f"Config path:   {CONFIG_PATH}")

print("\nTop-level sections:")
for section_name in raw_config:
    print(f"  - {section_name}")

print("\nYAML loaded as a Python dict:")
pprint(raw_config, sort_dicts=False)

Project root: c:\Users\yanzhang\Desktop\Research_Projects\radar_basics
Config path:   c:\Users\yanzhang\Desktop\Research_Projects\radar_basics\example_configs\architecture.yaml

Top-level sections:
  - radar
  - array
  - waveform
  - scan
  - scene
  - processing
  - run

YAML loaded as a Python dict:
{'radar': {'carrier_frequency_hz': 10000000000.0,
           'peak_tx_power_w': 50000.0,
           'noise_figure_db': 0.0,
           'system_loss_db': 0.0,
           'temperature_k': 0.0},
 'array': {'num_y': 4, 'num_x': 4},
 'waveform': {'bandwidth_hz': 5000000.0,
              'pulse_width_s': 2e-06,
              'prf_hz': 10000.0,
              'num_pulses': 16,
              'sample_rate_hz': 10000000.0},
 'scan': {'azimuths_deg': [-10.0, 0.0, 10.0],
          'elevations_deg': [-5.0, 0.0, 5.0],
          'mode': 'search'},
 'scene': {'targets': [{'name': 'target-a',
                        'range_m': 1500.0,
                        'az_deg': 0.0,
                        'el_deg'

### 观察这个输出

请先重点看 top-level sections，而不是陷入每个数字的细节。

- `radar`：雷达本机参数，比如 carrier frequency、peak transmit power、noise/loss。
- `array`：阵列几何，当前是二维 rectangular planar array。
- `waveform`：LFM pulse waveform 和 CPI 相关参数。
- `scan`：雷达要看哪些 azimuth/elevation 方向。
- `scene`：truth world，里面有哪些目标。
- `processing`：处理链路的角度网格、detector、tracker 配置。
- `run`：这次实验跑几轮 scan、随机种子、是否保存 raw IQ。

暂停点：运行上面的 cell 后，确认你能回答这个问题：如果我只改 YAML 里的 `scene.targets[0].range_m`，我是在改雷达硬件、改目标真值，还是改信号处理算法？

## Step 2: Config -> Python Objects

Step 1 里，`yaml.safe_load()` 得到的是普通 nested dictionary。Python 可以处理它，但这个 dict 还没有被这个 codebase 理解成一个正式的 radar experiment。

现在我们用 `load_config()` 做下一层转换：

```text
YAML file -> nested dict -> ExperimentConfig dataclass
```

`ExperimentConfig` 的价值是：它把 loose 的 dict 变成有结构、有字段名、经过基本校验的 Python object。

In [ ]:
from dataclasses import fields, is_dataclass

from radar_basics import load_config


config = load_config(CONFIG_PATH)

print(f"raw_config type: {type(raw_config).__name__}")
print(f"config type:     {type(config).__name__}")
print(f"is dataclass:    {is_dataclass(config)}")

print("\nTop-level config objects:")
for field in fields(config):
    value = getattr(config, field.name)
    print(f"  {field.name:<10} -> {type(value).__name__}")

print("\nA few concrete fields:")
print(f"carrier_frequency_hz = {config.radar.carrier_frequency_hz:,.1f}")
print(f"array size           = {config.array.num_y} x {config.array.num_x}")
print(f"num_pulses           = {config.waveform.num_pulses}")
print(f"scan azimuths        = {config.scan.azimuths_deg}")
print(f"scan elevations      = {config.scan.elevations_deg}")
print(f"target count         = {len(config.scene.targets)}")
print(f"run scan cycles      = {config.run.num_scan_cycles}")

### 观察这个输出

这里有一个重要分层：

- `raw_config` 是普通 `dict`，来自 YAML parser。
- `config` 是 `ExperimentConfig`，来自 `radar_basics.config.load_config()`。
- `config.radar`、`config.array`、`config.waveform` 等字段也各自是 dataclass object。

这一步还没有创建真正会参与仿真的 runtime objects。比如现在的 `config.radar` 只是雷达配置；下一步 Step 3A 里，我们会用它 build 出真正的 `RadarSystem`。

暂停点：请确认你能区分这三层：YAML text、Python dict、`ExperimentConfig` dataclass。

### My understanding after Step 2

到目前为止，可以把这三种形式理解成同一份 simulation specification 的不同表示：

```text
YAML text -> raw_config dictionary -> ExperimentConfig dataclass
```

它们表达的信息本质上是一样的：都是这次 radar simulation 需要的参数。区别主要在数据组织方式：

- YAML 方便人类阅读和编辑。
- `raw_config` 是 Python 可以直接处理的 nested `dict` / `list`。
- `ExperimentConfig` 是更结构化的 dataclass object，把 `radar`、`array`、`waveform`、`scan`、`scene`、`processing`、`run` 分别放进对应的 config dataclass。

这个 dataclass abstraction 对小项目来说看起来有点重，但它提供了字段结构、基本校验和更清楚的代码接口。

真正重要的边界是下一步：

```text
ExperimentConfig -> RadarSystem / Scene / Scheduler
```

从这里开始，就不只是同一份参数换一种组织方式了，而是会 build 出有行为的 runtime objects。比如 `RadarSystem` 可以计算 wavelength/range resolution，`Scene` 可以在某个时间产生 target snapshot，`Scheduler` 可以生成 dwell tasks。

## Step 3A: Radar Module

现在开始从 config specification 进入 runtime objects。第一站是 `radar_basics.radar`。

这一步我们只学习 radar 这一层：

- `config.radar`、`config.array`、`config.waveform` 是配置参数。
- `build_radar_system(config)` 会把这些参数 build 成一个 `RadarSystem`。
- `RadarSystem` 里面包含真正参与仿真的 `RectangularArray` 和 `LfmPulseWaveform`。

Mental model: `RadarSystem` 代表“这台雷达是什么样的”。它不包含目标，也不包含扫描任务；它只描述硬件、阵列、波形和由这些参数推导出来的雷达能力。

In [ ]:
from radar_basics import build_radar_system


radar = build_radar_system(config)
array = radar.array
waveform = radar.waveform

print("Config objects -> runtime object")
print(f"config.radar type:    {type(config.radar).__name__}")
print(f"config.array type:    {type(config.array).__name__}")
print(f"config.waveform type: {type(config.waveform).__name__}")
print(f"radar type:           {type(radar).__name__}")

print("\nRadarSystem: derived quantities")
print(f"carrier_frequency_hz       = {radar.carrier_frequency_hz:,.1f} Hz")
print(f"wavelength_m               = {radar.wavelength_m:.6f} m")
print(f"max_unambiguous_range_m    = {radar.max_unambiguous_range_m:,.2f} m")
print(f"range_resolution_m         = {radar.range_resolution_m:.2f} m")
print(f"velocity_resolution_mps    = {radar.velocity_resolution_mps:.2f} m/s")
print(f"max_unambiguous_velocity_mps = {radar.max_unambiguous_velocity_mps:.2f} m/s")
print(f"noise_power_w              = {radar.noise_power_w:.3e} W")

print("\nRectangularArray: element geometry")
print(f"array type        = {type(array).__name__}")
print(f"num_y x num_x     = {array.num_y} x {array.num_x}")
print(f"num_elements      = {array.num_elements}")
print(f"spacing_y_m       = {array.spacing_y_m:.6f} m")
print(f"spacing_x_m       = {array.spacing_x_m:.6f} m")
print(f"positions_m shape = {array.positions_m.shape}")
print(f"corner element 0  = {array.positions_m[0, 0]}")
print(f"corner element -1 = {array.positions_m[-1, -1]}")

print("\nLfmPulseWaveform: pulse and CPI settings")
print(f"waveform type          = {type(waveform).__name__}")
print(f"bandwidth_hz           = {waveform.bandwidth_hz:,.1f} Hz")
print(f"pulse_width_s          = {waveform.pulse_width_s:.3e} s")
print(f"prf_hz                 = {waveform.prf_hz:,.1f} Hz")
print(f"pri_s                  = {waveform.pri_s:.3e} s")
print(f"num_pulses             = {waveform.num_pulses}")
print(f"sample_rate_hz         = {waveform.sample_rate_hz:,.1f} Hz")
print(f"num_fast_time_samples  = {waveform.num_fast_time_samples}")
print(f"num_pulse_samples      = {waveform.num_pulse_samples}")

### 观察这个输出

这里开始出现真正的 runtime behavior：

- `config.radar` 是 `RadarConfig`，只是参数容器。
- `radar` 是 `RadarSystem`，它可以根据参数计算 `wavelength_m`、`range_resolution_m`、`velocity_resolution_mps` 等派生量。
- `radar.array` 是 `RectangularArray`，它知道阵元数量、间距和每个阵元的三维坐标。
- `radar.waveform` 是 `LfmPulseWaveform`，它知道 PRI、fast-time samples、pulse samples 等波形行为。

请先不要背公式，先抓住这个边界：YAML/config 只保存参数；`RadarSystem` 开始把参数变成可以计算、可以参与仿真的对象。

暂停点：请确认你能解释 `config.waveform.num_pulses` 和 `radar.waveform.num_pulses` 的关系，以及为什么 `radar.wavelength_m` 不需要直接写在 YAML 里。

### My understanding after Step 3A

Step 3A 里的 `radar` object 表示的是硬件层面的雷达系统，或者更准确地说，是一个 hardware + waveform runtime model。

它主要包含三类东西：

1. Antenna array / `RectangularArray`

   阵列对象保存阵元布局信息，比如 `num_y`、`num_x`、阵元间距，以及每个 element 的三维位置 `positions_m`。

   更准确地说，steering vector 不是提前存储在对象里的表，而是通过阵列几何和给定方向按需计算出来的：

   ```python
   array.steering_vector(wavelength_m, az_deg, el_deg)
   ```

   `transmit_field_gain()` 也是类似的 runtime calculation：根据 beam look direction 和 target direction 计算 transmit field gain。

2. Waveform / `LfmPulseWaveform`

   波形对象保存 LFM pulse 的基础参数，比如 bandwidth、pulse width、PRF、number of pulses 和 sample rate。

   它也能根据这些参数推导出 PRI、chirp slope、fast-time axis、pulse-time axis，以及 chirp sample values。

   同样地，chirp samples 不是 YAML 里直接存的东西，而是通过方法按需生成：

   ```python
   waveform.samples()
   ```

3. Radar performance specs / derived quantities

   `RadarSystem` 还根据 carrier frequency、waveform、noise/loss 等参数计算一些雷达性能指标，比如 wavelength、max unambiguous range、range resolution、velocity resolution、max unambiguous velocity 和 noise power。

所以我现在的理解是：

```text
config.radar/config.array/config.waveform = 参数说明
radar = 可以根据这些参数计算阵列响应、波形采样和雷达性能指标的 runtime object
```